In [1]:
import scanpy as sc
import pandas as pd
import os
from scipy.io import mmread
from scipy import sparse
import numpy as np
import anndata

In [5]:
def make_block_matrix(a, n, block_value=1, other_value=2):
    """
    Return an (a x a*n) matrix where for row i (0-based)
    columns [i*n, i*n + n - 1] are set to block_value,
    all other entries set to other_value.
    """
    b = a * n
    M = np.full((a, b), other_value, dtype=int)
    for i in range(a):
        start = i * n
        M[i, start:start + n] = block_value
    return M

In [29]:
def pseudobulk_from_anndata(adata,
                            groupby,            # e.g. "cell_type" (one value per cell)
                            n_pseudobulks=5,    # per group if using random-subsets
                            cells_per_pb=200,   # how many single cells to sum per pseudobulk
                            suffix = True,
                            random_seed=0):
    """
    Return: pandas DataFrame with index = genes (adata.var_names), columns = pseudobulk samples.
    Values are raw counts sums (numeric).
    """
    np.random.seed(random_seed)
    X = adata.X
    genes = adata.var_names.tolist()

    # ensure obs has groupby
    if groupby not in adata.obs.columns:
        raise KeyError(f"{groupby} not in adata.obs")

    pb_cols = {}
    # iterate groups
    groups = adata.obs[groupby].unique().tolist()
    for g in groups:
        mask = (adata.obs[groupby] == g).values
        idx = np.nonzero(mask)[0]
        if len(idx) == 0:
            continue

        cell_indices = idx.tolist()
        for i in range(n_pseudobulks):
            replace = len(cell_indices) < cells_per_pb
            chosen = (np.random.choice(cell_indices, size=cells_per_pb, replace=replace)
                        if cells_per_pb > 0 else np.array(cell_indices))
            if sparse.issparse(X):
                vec = X[chosen, :].sum(axis=0)
                vec = np.asarray(vec).ravel()
            else:
                vec = X[chosen, :].sum(axis=0)
                vec = np.asarray(vec).ravel()
            colname = f"{g}__pb{i+1}"
            pb_cols[colname] = vec

    
    class_mat = make_block_matrix(len(groups), n_pseudobulks)
    class_df = pd.DataFrame(class_mat,
                            index=groups)

    # make DataFrame: rows = genes, cols = pseudobulks
    if len(pb_cols) == 0:
        raise RuntimeError("No pseudobulks produced. Check groupby column and adata.obs.")
    pb_df = pd.DataFrame(pb_cols, index=genes)
    # ensure numeric dtype
    pb_df = pb_df.fillna(0).astype(float)
    if not suffix:
        pb_df.columns = [i.split('__')[0] for i in pb_df.columns]
    return pb_df,class_df

In [3]:
adata_public = sc.read_h5ad(r'E:\AAA_Labwork\Tcell model RNAseq\1_Healthy_Pan-GI_atlas_all_lineages_20241119.h5ad') #that's the data we got from gut cell atlas

In [8]:
adata_public_colon_epithelium = adata_public[(adata_public.obs['organ_groups'] == 'Large_intestine') & (adata_public.obs['tissue_fraction'].isin([ 'Epithelium']))]

In [9]:
adata_public_colon_epithelium.obs['level_1_annot']

index
AAACCTGAGACAGACC-GSM4766846    T and NK cells
AAACCTGGTTCGCTAA-GSM4766846    T and NK cells
AAACCTGTCTGAGGGA-GSM4766846    T and NK cells
AAACGGGAGAAGGACA-GSM4766846    T and NK cells
AAACGGGAGCAGATCG-GSM4766846    T and NK cells
                                    ...      
TTTGTCAAGTGTCCAT-GSM3587013    T and NK cells
TTTGTCACACAGACAG-GSM3587013        Epithelial
TTTGTCACACAGACTT-GSM3587013        Epithelial
TTTGTCATCACGACTA-GSM3587013        Epithelial
TTTGTCATCCTTTACA-GSM3587013        Epithelial
Name: level_1_annot, Length: 36737, dtype: category
Categories (6, object): ['B and B plasma', 'Endothelial', 'Epithelial', 'Mesenchymal', 'Myeloid', 'T and NK cells']

In [10]:
adata_public_colon_epithelium = adata_public_colon_epithelium[adata_public_colon_epithelium.obs['level_1_annot'].isin(['Mesenchymal','Endothelial'])==0, :]

In [11]:
adata_public_colon_epithelium.obs['level_1_annot']

index
AAACCTGAGACAGACC-GSM4766846    T and NK cells
AAACCTGGTTCGCTAA-GSM4766846    T and NK cells
AAACCTGTCTGAGGGA-GSM4766846    T and NK cells
AAACGGGAGAAGGACA-GSM4766846    T and NK cells
AAACGGGAGCAGATCG-GSM4766846    T and NK cells
                                    ...      
TTTGTCAAGTGTCCAT-GSM3587013    T and NK cells
TTTGTCACACAGACAG-GSM3587013        Epithelial
TTTGTCACACAGACTT-GSM3587013        Epithelial
TTTGTCATCACGACTA-GSM3587013        Epithelial
TTTGTCATCCTTTACA-GSM3587013        Epithelial
Name: level_1_annot, Length: 36694, dtype: category
Categories (4, object): ['B and B plasma', 'Epithelial', 'Myeloid', 'T and NK cells']

30, a threshold to safely use t-test in statistics, is chosen as the minimum number of replicates for a cell type signature to faithfully summarized.

In [12]:
gut_celltype = adata_public_colon_epithelium.obs['level_2_annot'].value_counts().index[adata_public_colon_epithelium.obs['level_2_annot'].value_counts()>=30]
gut_celltype

CategoricalIndex(['Absorptive', 'Secretory', 'Transit_amplifying',
                  'Conventional_CD4', 'Conventional_CD8',
                  'Unconventional_T/ILC', 'Stem', 'Treg', 'Enteroendocrine',
                  'Mature_B', 'NK', 'Cycling_T/NK'],
                 categories=['Absorptive', 'B_plasma', 'Conventional_CD4', 'Conventional_CD8', ..., 'Stem', 'Transit_amplifying', 'Treg', 'Unconventional_T/ILC'], ordered=False, dtype='category', name='level_2_annot')

In [14]:
adata_public_colon_epithelium = adata_public_colon_epithelium[adata_public_colon_epithelium.obs['level_2_annot'].isin(gut_celltype),:]

In [15]:
adata_public_colon_epithelium.obs['level_2_annot'].values.categories

Index(['Absorptive', 'Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK',
       'Enteroendocrine', 'Mature_B', 'NK', 'Secretory', 'Stem',
       'Transit_amplifying', 'Treg', 'Unconventional_T/ILC'],
      dtype='object')

specifically I would argue that B can be removed for possibly inappropriate automated annotation or multiplet or incomplete separation between the epithelium and lamina propira.

In [16]:
filtered_celltypes = ['B and B plasma']

In [17]:
adata_public_colon_epithelium_filtered = adata_public_colon_epithelium[adata_public_colon_epithelium.obs['level_1_annot'].isin(filtered_celltypes)==False,:]

In [18]:
adata_public_colon_epithelium_filtered

View of AnnData object with n_obs × n_vars = 36545 × 36390
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'technology', 'disease', 'sex'
    var: 'gene_ids'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'control_vs_disease_colors', 'default_embedding', 'level_1_annot_colors', 'level_2_annot_colors', 'level_3_annot_colors', 'neighbors', 'neighbors_scvi', 'organ_groups_colors', 'organ_unified_colors', 'sample_disease_colors', 'study_colors', 'title', 'umap'
    obsm: 'X_scANVI', 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'neighbors_scvi_connectivities', 'neighbors_scvi_distances'

In [19]:
adata_public_colon_epithelium_filtered.obs['level_1_annot'].values.categories

Index(['Epithelial', 'T and NK cells'], dtype='object')

In [20]:
adata_public_colon_epithelium_filtered.obs['level_2_annot'].values.categories

Index(['Absorptive', 'Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK',
       'Enteroendocrine', 'NK', 'Secretory', 'Stem', 'Transit_amplifying',
       'Treg', 'Unconventional_T/ILC'],
      dtype='object')

In [21]:
adata_public_colon_epithelium_filtered.obs['level_3_annot'].values.categories

Index(['BEST4_enterocyte_colonocyte', 'Colonocyte', 'DCS_MUC17', 'Enterocyte',
       'Enteroendocrine', 'Enteroendocrine_progenitor', 'Epithelial_stem',
       'Goblet', 'ILC3', 'MAIT', 'Mature_colonocyte', 'NK_CD16',
       'NK_CD56bright', 'T/NK_cycling', 'TA', 'Tfh', 'Tfh_naive',
       'Tnaive/cm_CD4', 'Tnaive/cm_CD8', 'Treg', 'Treg_IL10', 'Trm/em_CD8',
       'Trm_CD4', 'Trm_CD8', 'Trm_Th17', 'Tuft', 'gdT', 'gdT_naive'],
      dtype='object')

In [22]:
non_immune_celltype = ['Absorptive', 
       'Enteroendocrine', 'Secretory', 'Stem', 'Transit_amplifying']

In [23]:
epi_immune = adata_public_colon_epithelium_filtered[adata_public_colon_epithelium_filtered.obs['level_2_annot'].isin(non_immune_celltype) ==False, :]
epi_nonimmune = adata_public_colon_epithelium_filtered[adata_public_colon_epithelium_filtered.obs['level_2_annot'].isin(non_immune_celltype) ==True, :]

In [24]:
epi_nonimmune

View of AnnData object with n_obs × n_vars = 25427 × 36390
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'technology', 'disease', 'sex'
    var: 'gene_ids'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'control_vs_disease_colors', 'default_embedding', 'level_1_annot_colors', 'level_2_annot_colors', 'level_3_annot_colors', 'neighbors', 'neighbors_scvi', 'organ_groups_colors', 'organ_unified_colors', 'sample_disease_colors', 'study_colors', 'title', 'umap'
    obsm: 'X_scANVI', 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'neighbors_scvi_connectivities', 'neighbors_scvi_distances'

In [25]:
epi_nonimmune.obs['level_2_annot'].values.categories

Index(['Absorptive', 'Enteroendocrine', 'Secretory', 'Stem',
       'Transit_amplifying'],
      dtype='object')

In [26]:
epi_immune

View of AnnData object with n_obs × n_vars = 11118 × 36390
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'technology', 'disease', 'sex'
    var: 'gene_ids'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'control_vs_disease_colors', 'default_embedding', 'level_1_annot_colors', 'level_2_annot_colors', 'level_3_annot_colors', 'neighbors', 'neighbors_scvi', 'organ_groups_colors', 'organ_unified_colors', 'sample_disease_colors', 'study_colors', 'title', 'umap'
    obsm: 'X_scANVI', 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'neighbors_scvi_connectivities', 'neighbors_scvi_distances'

In [27]:
epi_immune.obs['level_2_annot'].values.categories

Index(['Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK', 'NK', 'Treg',
       'Unconventional_T/ILC'],
      dtype='object')

In [ ]:
epi_immune_subsampled = sc.pp.subsample(epi_immune, n_obs=5000, random_state=0, copy=True)
counts = epi_immune_subsampled.layers['counts']
try:
    mat = counts.T.todense()   # works for sparse matrices
except Exception:
    mat = np.asarray(counts).T # fallback for dense arrays

df_epi_immune = pd.DataFrame(mat,
                             index=epi_immune_subsampled.var_names,
                             columns=epi_immune_subsampled.obs['level_2_annot'].values)  # allow duplicate colnames
df_epi_immune.index.name = 'GeneSymbol'

In [ ]:
df_epi_immune.to_csv(r"C:\Users\16220\Documents\GitHub\gut-liver-model\RNAseq\deconvolution\scRNA_ref\public\epi_immune_ref.txt",sep ='\t', index=True)

In [85]:
epi_nonimmune_subsampled = sc.pp.subsample(epi_nonimmune, n_obs=5000, random_state=0, copy=True)
counts = epi_nonimmune_subsampled.layers['counts']
try:
    mat = counts.T.todense()   # works for sparse matrices
except Exception:
    mat = np.asarray(counts).T # fallback for dense arrays

df_epi_nonimmune = pd.DataFrame(mat,
                             index=epi_nonimmune_subsampled.var_names,
                             columns=epi_nonimmune_subsampled.obs['level_2_annot'].values)  # allow duplicate colnames
df_epi_nonimmune.index.name = 'GeneSymbol'

In [ ]:
df_epi_nonimmune.to_csv(r"C:\Users\16220\Documents\GitHub\gut-liver-model\RNAseq\deconvolution\scRNA_ref\public\epi_nonimmune_ref.txt",sep ='\t', index=True)

In [88]:
epi_all_subsampled = sc.pp.subsample(adata_public_colon_epithelium_filtered, n_obs=5000, random_state=0, copy=True)
counts = epi_all_subsampled.layers['counts']
try:
    mat = counts.T.todense()   # works for sparse matrices
except Exception:
    mat = np.asarray(counts).T # fallback for dense arrays

df_epi_all = pd.DataFrame(mat,
                             index=epi_all_subsampled.var_names,
                             columns=epi_all_subsampled.obs['level_2_annot'].values)  # allow duplicate colnames
df_epi_all.index.name = 'GeneSymbol'

In [ ]:
df_epi_all.to_csv(r"C:\Users\16220\Documents\GitHub\gut-liver-model\RNAseq\deconvolution\scRNA_ref\public\epi_all_ref.txt",sep ='\t', index=True)

try pseudobulk as well

In [31]:
adata_list = [adata_public_colon_epithelium_filtered, epi_immune, epi_nonimmune]
adata_names = ['epi_all', 'epi_immune', 'epi_nonimmune']
for i in range(len(adata_list)):
    adata = adata_list[i]
    name = adata_names[i]
    adata.X = adata.layers['counts']
    pb, class_df = pseudobulk_from_anndata(adata, groupby='level_2_annot', n_pseudobulks=10, cells_per_pb=100, suffix= False, random_seed=0)
    pb.index.name = 'GeneSymbol'
    pb.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\{name}_pb.txt',sep='\t')
    class_df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\{name}_class_df.txt',sep='\t',header = False)

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_29852\1485508286.py:6: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata.X = adata.layers['counts']
c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.


then the lp, which is only cd45+

In [32]:
adata_public.obs['tissue_fraction'].values

['Mucosa', 'Mucosa', 'Mucosa', 'Mucosa', 'Mucosa', ..., 'Mucosa', 'Mucosa', 'Mucosa', 'Mucosa', 'Mucosa']
Length: 1077241
Categories (4, object): ['Epithelium', 'Full_thickness', 'Lamina_propria', 'Mucosa']

In [33]:
adata_public_LP = adata_public[(adata_public.obs['organ_groups'] == 'Large_intestine') & (adata_public.obs['tissue_fraction'].isin(['Mucosa','Lamina_propria']))]
adata_public_LP.obs['level_1_annot']

index
AAACCTGAGATGCGAC-GSM3584928    T and NK cells
AAACCTGAGGAATTAC-GSM3584928    T and NK cells
AAACCTGCAGATGGGT-GSM3584928    T and NK cells
AAACCTGGTTCGGGCT-GSM3584928    T and NK cells
AAACCTGGTTTGCATG-GSM3584928    T and NK cells
                                    ...      
TTTGTCAAGAACAACT-GSM5525958    T and NK cells
TTTGTCAAGATAGGAG-GSM5525958    B and B plasma
TTTGTCAAGTAGGCCA-GSM5525958    B and B plasma
TTTGTCAGTTGGACCC-GSM5525958    T and NK cells
TTTGTCATCACATACG-GSM5525958    B and B plasma
Name: level_1_annot, Length: 79174, dtype: category
Categories (7, object): ['B and B plasma', 'Endothelial', 'Epithelial', 'Mesenchymal', 'Myeloid', 'Neural', 'T and NK cells']

In [34]:
adata_public_LP = adata_public_LP[adata_public_LP.obs['level_1_annot'].isin(['Neural','Epithelial','Mesenchymal','Endothelial'])==0, :]
adata_public_LP.obs['level_1_annot']

index
AAACCTGAGATGCGAC-GSM3584928    T and NK cells
AAACCTGAGGAATTAC-GSM3584928    T and NK cells
AAACCTGCAGATGGGT-GSM3584928    T and NK cells
AAACCTGGTTCGGGCT-GSM3584928    T and NK cells
AAACCTGGTTTGCATG-GSM3584928    T and NK cells
                                    ...      
TTTGTCAAGAACAACT-GSM5525958    T and NK cells
TTTGTCAAGATAGGAG-GSM5525958    B and B plasma
TTTGTCAAGTAGGCCA-GSM5525958    B and B plasma
TTTGTCAGTTGGACCC-GSM5525958    T and NK cells
TTTGTCATCACATACG-GSM5525958    B and B plasma
Name: level_1_annot, Length: 58223, dtype: category
Categories (3, object): ['B and B plasma', 'Myeloid', 'T and NK cells']

In [35]:
LP_celltype = adata_public_LP.obs['level_2_annot'].value_counts().index[adata_public_LP.obs['level_2_annot'].value_counts()>=30]
LP_celltype

CategoricalIndex(['Mature_B', 'Conventional_CD4', 'B_plasma',
                  'Conventional_CD8', 'Unconventional_T/ILC', 'Treg',
                  'Macrophage', 'NK', 'DC', 'Monocyte', 'Granulocyte',
                  'Cycling_T/NK'],
                 categories=['B_plasma', 'Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK', ..., 'Monocyte', 'NK', 'Treg', 'Unconventional_T/ILC'], ordered=False, dtype='category', name='level_2_annot')

In [36]:
adata_public_LP = adata_public_LP[adata_public_LP.obs['level_2_annot'].isin(LP_celltype),:]
adata_public_LP.obs['level_2_annot'].values.categories

Index(['B_plasma', 'Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK',
       'DC', 'Granulocyte', 'Macrophage', 'Mature_B', 'Monocyte', 'NK', 'Treg',
       'Unconventional_T/ILC'],
      dtype='object')

In [37]:
filtered_celltypes = ['Granulocyte']
adata_public_LP_filtered = adata_public_LP[adata_public_LP.obs['level_2_annot'].isin(filtered_celltypes)==False,:]
adata_public_LP_filtered

View of AnnData object with n_obs × n_vars = 57850 × 36390
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'technology', 'disease', 'sex'
    var: 'gene_ids'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'control_vs_disease_colors', 'default_embedding', 'level_1_annot_colors', 'level_2_annot_colors', 'level_3_annot_colors', 'neighbors', 'neighbors_scvi', 'organ_groups_colors', 'organ_unified_colors', 'sample_disease_colors', 'study_colors', 'title', 'umap'
    obsm: 'X_scANVI', 'X_scvi', 'X_umap'
    layers: 'counts'
    obsp: 'neighbors_scvi_connectivities', 'neighbors_scvi_distances'

In [38]:
adata_public_LP_filtered.obs['level_2_annot'].values.categories

Index(['B_plasma', 'Conventional_CD4', 'Conventional_CD8', 'Cycling_T/NK',
       'DC', 'Macrophage', 'Mature_B', 'Monocyte', 'NK', 'Treg',
       'Unconventional_T/ILC'],
      dtype='object')

In [38]:
LP_immune_subsampled = sc.pp.subsample(adata_public_LP_filtered, n_obs=5000, random_state=0, copy=True)
counts = LP_immune_subsampled.layers['counts']
try:
    mat = counts.T.todense()   # works for sparse matrices
except Exception:
    mat = np.asarray(counts).T # fallback for dense arrays

df_LP_immune = pd.DataFrame(mat,
                             index=LP_immune_subsampled.var_names,
                             columns=LP_immune_subsampled.obs['level_2_annot'].values)  # allow duplicate colnames

In [40]:
df_LP_immune.index.name = 'GeneSymbol'

In [41]:
df_LP_immune

,Conventional_CD4,B_plasma,Monocyte,Mature_B,Mature_B,Mature_B,Mature_B,Conventional_CD4,Conventional_CD8,Conventional_CD4,...,Mature_B,B_plasma,Mature_B,B_plasma,Cycling_T/NK,Treg,Unconventional_T/ILC,Mature_B,Conventional_CD4,B_plasma
GeneSymbol,,,,,,,,,,,,,,,,,,,,,
MIR1302-2HG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FAM138A,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
OR4F5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AL627309.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AL627309.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AC141272.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AC023491.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AC007325.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_LP_immune.to_csv(r"C:\Users\16220\Documents\GitHub\gut-liver-model\RNAseq\deconvolution\scRNA_ref\public\LP_immune_ref.txt",sep ='\t', index=True)

In [39]:
adata_public_LP_filtered.X = adata_public_LP_filtered.layers['counts']
pb, class_df = pseudobulk_from_anndata(adata_public_LP_filtered, groupby='level_2_annot', n_pseudobulks=10, cells_per_pb=100, suffix= False, random_seed=0)
pb.index.name = 'GeneSymbol'
pb.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\public_LP_filtered_pb.txt',sep='\t')
class_df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\public_LP_filtered_class_df.txt',sep='\t',header = False)

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\anndata\_core\anndata.py:618: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_29852\415097932.py:1: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata_public_LP_filtered.X = adata_public_LP_filtered.layers['counts']


For liver data, we got the reference from human liver atlas
somehow their full healthy human liver data feature.tsv has only 1 column, I'll just mannually read it

In [40]:
def pick(path, *names):
    for n in names:
        p = os.path.join(path, n)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of {names} found under {path}")

path = r"E:\AAA_Labwork\Tcell model RNAseq\liver_ref\rawData_human\countTable_human"
mtx_fp  = pick(path, "matrix.mtx.gz", "matrix.mtx")
bc_fp   = pick(path, "barcodes.tsv.gz", "barcodes.tsv")
feat_fp = pick(path, "features.tsv.gz", "features.tsv", "genes.tsv.gz", "genes.tsv")
feat = pd.read_csv(feat_fp, header=None, sep='\t', engine="python")
bc = pd.read_csv(bc_fp, header=None, sep='\t', engine="python")

In [41]:
X = mmread(mtx_fp)            # scipy sparse
if not sparse.issparse(X):
    X = sparse.csr_matrix(X)
X = X.tocsr().T

In [42]:
assert X.shape[0] == len(bc), f"Cells mismatch: {X.shape[0]} vs {len(bc)}"
assert X.shape[1] == len(feat), f"Genes mismatch: {X.shape[1]} vs {len(feat)}"

In [43]:
adata = anndata.AnnData(X, dtype=X.dtype)
adata.obs_names = pd.Index(bc.values.flatten(), name="barcode")
adata.var_names = pd.Index(feat.values.flatten(), name="gene_symbol")

c:\Users\16220\anaconda3\envs\scvi-env\Lib\site-packages\legacy_api_wrap\__init__.py:82: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


In [25]:
adata.write_h5ad(r'E:\AAA_Labwork\Tcell model RNAseq\liver_ref\rawData_human\live_full.h5ad')

In [5]:
adata = sc.read_h5ad(r'E:\AAA_Labwork\Tcell model RNAseq\liver_ref\rawData_human\live_full.h5ad')

In [44]:
adata

AnnData object with n_obs × n_vars = 272867 × 32738

In [45]:
annot = pd.read_csv(r'E:\AAA_Labwork\Tcell model RNAseq\liver_ref\rawData_human\annot_humanAll.csv', index_col='cell')

In [46]:
annot

,UMAP_1,UMAP_2,cluster,annot,sample,patient,digest,typeSample,diet
cell,,,,,,,,,
AAACCTGTCAGGATCT-1,0.738062,-3.097089,34,Mono+mono derived cells,CISE06,H02,exVivo,citeSeq,Lean
AAACGGGAGCACAGGT-1,0.367818,-3.251462,34,Mono+mono derived cells,CISE06,H02,exVivo,citeSeq,Lean
AAACGGGAGTGTGGCA-1,-0.728939,-5.994781,14,Mono+mono derived cells,CISE06,H02,exVivo,citeSeq,Lean
AAACGGGAGTTAGGTA-1,2.814624,-7.131484,7,Mono+mono derived cells,CISE06,H02,exVivo,citeSeq,Lean
AAACGGGCACCAGATT-1,3.420414,-6.966918,7,Mono+mono derived cells,CISE06,H02,exVivo,citeSeq,Lean
...,...,...,...,...,...,...,...,...,...
TTTACCAAGAGAGGTA-41,-3.674058,7.171934,41,Fibroblasts,CS171,H38,exVivo,scRnaSeq,Lean
TTTACCAGTATCGAAA-41,-4.356690,7.339135,41,Fibroblasts,CS171,H38,exVivo,scRnaSeq,Lean
TTTCCTCGTGAGGCAT-41,-3.759264,7.485170,41,Fibroblasts,CS171,H38,exVivo,scRnaSeq,Lean


In [47]:
common_bc = adata.obs_names.intersection(annot.index)
len(common_bc)

167598

In [48]:
adata= adata[common_bc,:]

In [49]:
adata.obs = annot.loc[common_bc,['annot','patient','sample','digest','typeSample','diet']]

In [50]:
set(adata.obs['annot'].values)

{'B cells',
 'Basophils',
 'Cholangiocytes',
 'Circulating NK/NKT',
 'Endothelial cells',
 'Fibroblasts',
 'Hepatocytes',
 'Macrophages',
 'Mig.cDCs',
 'Mono+mono derived cells',
 'Neutrophils',
 'Plasma cells',
 'Resident NK',
 'T cells',
 'cDC1s',
 'cDC2s',
 'pDCs'}

In [51]:
liver_immune = adata[adata.obs['annot'].isin(['Cholangiocytes','Endothelial cells','Fibroblasts','Hepatocytes'])==0 , :]

In [52]:
liver_immune

View of AnnData object with n_obs × n_vars = 151838 × 32738
    obs: 'annot', 'patient', 'sample', 'digest', 'typeSample', 'diet'

In [17]:
set(liver_immune.obs['annot'].values)

{'B cells',
 'Basophils',
 'Circulating NK/NKT',
 'Macrophages',
 'Mig.cDCs',
 'Mono+mono derived cells',
 'Neutrophils',
 'Plasma cells',
 'Resident NK',
 'T cells',
 'cDC1s',
 'cDC2s',
 'pDCs'}

In [53]:
liver_immune_subsampled = sc.pp.subsample(liver_immune, n_obs=5000, random_state=0, copy=True)

In [19]:
df = pd.DataFrame.sparse.from_spmatrix(liver_immune.X.T,  # genes x cells
                                           index=liver_immune.var_names,
                                           columns=liver_immune.obs['annot'])

In [21]:
df.index.name = ''

In [ ]:
df.to_csv(r"C:\Users\16220\Documents\GitHub\gut-liver-model\RNAseq\deconvolution\scRNA_ref\public\liver_immune_ref.txt",sep ='\t', index=True)

In [55]:
pb, class_df = pseudobulk_from_anndata(liver_immune, groupby='annot', n_pseudobulks=10, cells_per_pb=100, suffix= False, random_seed=0)
pb.index.name = 'GeneSymbol'
pb.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\liver_immune_pb.txt',sep='\t')
class_df.to_csv(f'C:\\Users\\16220\\Documents\\GitHub\\gut-liver-model\\RNAseq\\deconvolution\\scRNA_ref\\public\\pb\\liver_immune_class_df.txt',sep='\t',header = False)